## DQ Gate - Between Ingestion and Transformation

Runs as the second task in `stripex-orchestration-job`, after ingestion, before transformation. This is a JOB-LEVEL health check, not a row-level quality rule — it asks "is this entire run healthy enough to build on top of," not "should this one row be trusted." Row-level rules already exist in Bronze and Silver and never block anything. This gate can, and does, on purpose.

Two checks:\
- 1. Every bronze table has at least one row from `run_date` specifically.
- 2. No bronze quarantine pair's rate, computed only over `run_date`'s rows, has spiked past a coarse threshold (50%). This check only exists to catch "something is catastrophically wrong with today's batch," not to replace real DQ reporting.

If either check fails, this notebook raises, the task fails, and — because job
tasks only run if their dependency succeeded by default — transformation never
starts. Nothing silently continues on top of a broken run.

In [0]:
from datetime import datetime, timezone

dbutils.widgets.text("catalog", "dev", "Target catalog")
# Standalone default (real "today") for interactive testing outside a job run.
# When this notebook runs as part of stripex-orchestration-job, the job
# parameter run_date (default {{job.start_time.iso_date}}) overrides this automatically
dbutils.widgets.text("run_date", datetime.now(timezone.utc).date().isoformat(), "Run date (YYYY-MM-DD)")

catalog = dbutils.widgets.get("catalog")
run_date = dbutils.widgets.get("run_date")

In [0]:
BRONZE_TABLES = [
    "bronze_orders", "bronze_order_items", "bronze_customers",
    "bronze_products", "bronze_categories", "bronze_clickstream", "bronze_inventory",
]

# All 7 bronze quarantine pairs — the only layer that has actually finished
# executing by the time this gate runs. Every pair carries _ingested_at
# forward from bronze unchanged
QUARANTINE_PAIRS = [
    ("bronze_orders_valid", "bronze_orders_quarantined"),
    ("bronze_order_items_valid", "bronze_order_items_quarantined"),
    ("bronze_customers_valid", "bronze_customers_quarantined"),
    ("bronze_products_valid", "bronze_products_quarantined"),
    ("bronze_categories_valid", "bronze_categories_quarantined"),
    ("bronze_clickstream_valid", "bronze_clickstream_quarantined"),
    ("bronze_inventory_valid", "bronze_inventory_quarantined")
]

QUARANTINE_RATE_THRESHOLD = 0.5

def date_filter() -> str:
    """Every table this notebook checks is Bronze — _ingested_at is always a
    real timestamp column here."""
    return f"date(_ingested_at) = '{run_date}'"

In [0]:
failures = []

for table in BRONZE_TABLES:
    count = spark.table(f"{catalog}.stripex.{table}") \
        .filter(date_filter()) \
        .count()
    if count == 0: 
        failures.append(f"{table} has 0 rows for run_date={run_date} - ingestion may have failed silently.")

for valid_table, quarantined_table in QUARANTINE_PAIRS:
    filter_expr = date_filter()
    valid_count = spark.table(f"{catalog}.stripex.{valid_table}").filter(filter_expr).count()
    quarantined_count = spark.table(f"{catalog}.stripex.{quarantined_table}").filter(filter_expr).count()
    total = valid_count + quarantined_count
    if total > 0:
        rate = quarantined_count / total
        if rate > QUARANTINE_RATE_THRESHOLD:
            failures.append(
                f"{quarantined_table} quarantine rate for run_date={run_date} is {rate:.0%} "
                f"(threshold {QUARANTINE_RATE_THRESHOLD:.0%}) - {quarantine_count} of {total} rows."
            )

In [0]:
if failures:
    message = f"DQ Gate failed for run_date={run_date}:\n" + "\n".join(f"  -  {f}" for f in failures)
    print(message)
    raise Exception(message)

print(f"DQ Gate passed for run_date={run_date} - {len(BRONZE_TABLES)} bronze tables checked, "
      f"{len(QUARANTINE_PAIRS)} quarantine pairs within threshold.")